# TUS RAG evaluation review
Run this notebook from `agentic_rag/`. It reads the selected questions and the resumable result cache. An empty cache means the model run has not completed.

In [ ]:
import html
import json
from collections import Counter
from pathlib import Path
from IPython.display import HTML, display

base = Path.cwd() / 'evaluation'
if not base.is_dir():
    base = Path.cwd()
selection = json.loads((base / 'selection.json').read_text(encoding='utf-8'))
questions = [json.loads(line) for line in (base / 'questions.jsonl').read_text(encoding='utf-8').splitlines()]
results_path = base / 'cache' / 'results.jsonl'
latest = {}
if results_path.exists():
    for line in results_path.read_text(encoding='utf-8').splitlines():
        row = json.loads(line)
        latest[row['id']] = row
results = [latest.get(question['id'], {'id': question['id'], 'status': 'pending', **question}) for question in questions]
selection, Counter(row['status'] for row in results)

In [ ]:
answered = [row for row in results if row.get('model_answer')]
known = [row for row in answered if row.get('correct') is not None]
def gold_location(row):
    parsed = (row.get('assessment', {}).get('parsed') or {})
    return parsed.get('gold_corpus_location') or (parsed.get('corpus_location') if row.get('correct') is True else None)
locations = Counter((row.get('assessment', {}).get('parsed') or {}).get('corpus_assessment', 'unassessed') for row in answered)
{
    'selected': len(questions),
    'answered': len(answered),
    'assessed': sum(row['status'] == 'complete' and bool((row.get('assessment', {}).get('parsed') or {})) for row in answered),
    'correct': sum(row['correct'] is True for row in known),
    'incorrect': sum(row['correct'] is False for row in known),
    'ambiguous_answer': len(answered) - len(known),
    'accuracy_on_extracted_answers': sum(row['correct'] is True for row in known) / len(known) if known else None,
    'verified_gold_locations': sum(bool(gold_location(row)) for row in answered),
    'model_answer_locations': sum(bool((row.get('assessment', {}).get('parsed') or {}).get('corpus_location')) for row in answered),
    'gold_review_flags': sum(bool((row.get('assessment', {}).get('parsed') or {}).get('gold_review_flag')) for row in answered),
    'model_answer_corpus_assessments': dict(locations),
}

In [ ]:
headers = ['ID', 'Status', 'Gold', 'Model', 'Correct', 'Queries', 'Uses evidence', 'Model source', 'Gold source']
lines = []
for row in results:
    assessment = (row.get('assessment', {}).get('parsed') or {})
    location = assessment.get('corpus_location') or {}
    gold = gold_location(row) or {}
    lines.append([row['id'], row['status'], row['gold_answer'], row.get('predicted_answer'), row.get('correct'),
                  ' | '.join(row.get('retrieval_queries', [])), assessment.get('answer_uses_evidence'),
                  f"{location.get('source')}:{location.get('line_start')}" if location else None,
                  f"{gold.get('source')}:{gold.get('line_start')}" if gold else None])
table = '<table><thead><tr>' + ''.join(f'<th>{html.escape(h)}</th>' for h in headers) + '</tr></thead><tbody>'
table += ''.join('<tr>' + ''.join(f'<td>{html.escape(str(value)) if value is not None else ''}</td>' for value in row) + '</tr>' for row in lines)
display(HTML(table + '</tbody></table>'))

## Single-question inspection
Change `question_id` to any ID in the table, then run the cell. The complete retrieved parent documents are shown below the answer.

In [ ]:
question_id = questions[0]['id']
row = next(item for item in results if item['id'] == question_id)
def block(title, value):
    display(HTML(f'<h3>{html.escape(title)}</h3><pre style="white-space:pre-wrap">{html.escape(str(value))}</pre>'))
block('Question and choices', row['question'] + '\n' + row['choices'])
block('Gold / model / correctness', f"{row['gold_answer']} / {row.get('predicted_answer')} / {row.get('correct')}")
block('Model answer', row.get('model_answer', row.get('error', 'Pending')))
block('Retrieval queries', row.get('retrieval_queries', []))
for call_index, call in enumerate(row.get('retrieval_calls', []), 1):
    for doc_index, document in enumerate(call['documents'], 1):
        block(f"Retrieved {call_index}.{doc_index}: {document['metadata']}", document['content'])
block('Pruned evidence shown to responder', row.get('pruned_evidence', []))
block('Assessment', json.dumps(row.get('assessment', {}), ensure_ascii=False, indent=2))
block('Corpus candidates', json.dumps(row.get('corpus_candidates', []), ensure_ascii=False, indent=2))

In [ ]:
unresolved = [row['id'] for row in answered if not gold_location(row)]
print(f'{len(unresolved)} questions have no verified data/ line location:')
print(', '.join(unresolved))